# First PyTorch Geometric Milestone on ASU Sol

## Playing with PyG in a GNN Sandbox

**Student:** Sileshi Hirpa  
**Research direction:** Graph Neural Networks, Structural Stability, and Scientific Machine Learning  
**Advisor:** [Professor Yixuan He  ](https://search.asu.edu/profile/5144530)  
**Platform:** ASU Sol Supercomputer  
**Working directory:** `/scratch/shirpa/gnn-stability/research/brainstorming`  
**Conda environment:** `gnn_env`  
**Jupyter kernel:** `Python (gnn_env_pyg)`  

---

## Purpose of This Notebook

Professor He suggested that I begin by playing with the PyTorch Geometric library and trying small GNN examples. This notebook documents my first successful PyG milestone on ASU Sol.

The goal of this first phase is not to train a full model yet. The goal is to confirm that I can:

1. Use the correct Python environment on Sol.
2. Load a real graph dataset using PyTorch Geometric.
3. Inspect the structure of a PyG graph object.
4. Build a small two-layer Graph Convolutional Network.
5. Run one forward pass through the model.
6. Interpret the output mathematically and conceptually.

This notebook serves as my beginner research log before moving to training, perturbation experiments, and structural stability analysis.

# 1. Research Context

Graph Neural Networks, or **GNNs**, are neural networks designed to work with graph-structured data.

A graph is commonly written as:

$$
G = (V, E)
$$

where:

- $V$ is the set of nodes.
- $E$ is the set of edges.

In this first experiment, I use the **Cora citation graph**.

In the Cora dataset:

- Nodes represent academic papers.
- Edges represent citation relationships between papers.
- Node features describe each paper.
- Node labels represent paper categories.

This first notebook is a foundation for later research on **GNN structural stability**, where I will study how a GNN behaves when the graph structure changes.

# 2. CRISP-DM Mapping

Although this is a small technical test, I can still organize it using the CRISP-DM mindset.

| CRISP-DM Phase | What It Means in This Notebook |
|---|---|
| Business / Research Understanding | Understand whether PyG works on Sol for a beginner GNN example. |
| Data Understanding | Load and inspect the Cora graph. |
| Data Preparation | Use PyG's prepared graph object with features, edges, labels, and masks. |
| Modeling | Build a small two-layer GCN. |
| Evaluation | Confirm that the forward pass produces the expected output shape. |
| Deployment / Documentation | Save the notebook and logs inside the Sol sandbox for future upgrade. |

At this stage, the evaluation is simple. I am not measuring accuracy yet. I am only confirming that the graph and model pipeline works correctly.

In [1]:
# Code, Environment Check
import sys
import numpy as np
import scipy
import torch
import torch_geometric

print("=== Environment Check ===")
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)

/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


=== Environment Check ===
Python executable: /scratch/shirpa/gnn-stability/miniconda3/envs/gnn_env/bin/python
Python version: 3.11.15 (main, Mar 11 2026, 17:20:07) [GCC 14.3.0]
NumPy: 1.26.4
SciPy: 1.17.1
PyTorch: 2.12.0+cu130
PyTorch Geometric: 2.7.0


## Environment Check Result

The environment check was successful.

The notebook is using the correct Python executable:

```text
/scratch/shirpa/gnn-stability/miniconda3/envs/gnn_env/bin/python


---

# Cell 6: Code, Check Device

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Available device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected in this session. CPU is sufficient for this beginner test.")

# 4. Device Note

For this first notebook, CPU is enough because the Cora dataset is small.

Later, when I begin training larger models or running repeated perturbation experiments, I can request GPU resources on Sol.

For now, the goal is only:

```text
Load graph → inspect graph → run GCN forward pass


---

# Cell 8: Markdown, Dataset Introduction


# 5. Dataset: Cora Citation Graph

The Cora dataset is a standard benchmark dataset for node classification.

The graph is represented as:

$$
G = (V, E)
$$

where:

- $V$ is the set of papers.
- $E$ is the set of citation links.

The node feature matrix is written as:

$$
X \in \mathbb{R}^{N \times F}
$$

where:

- $N$ is the number of nodes.
- $F$ is the number of node features.

The label vector is written as:

$$
y \in \mathbb{R}^{N}
$$

where each entry gives the class label for one node.

In [3]:
# Code, Load Cora
from pathlib import Path
from torch_geometric.datasets import Planetoid

dataset_root = Path("data/Planetoid")

dataset = Planetoid(root=str(dataset_root), name="Cora")
data = dataset[0]

print("=== Dataset Loaded ===")
print(dataset)
print()
print("=== Graph Object ===")
print(data)

=== Dataset Loaded ===
Cora()

=== Graph Object ===
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


# 6. Understanding the PyG Data Object

PyTorch Geometric stores the graph in a `Data` object.

The most important fields are:

| Field | Meaning |
|---|---|
| `data.x` | Node feature matrix |
| `data.edge_index` | Graph connectivity |
| `data.y` | Node labels |
| `data.train_mask` | Nodes used for training |
| `data.val_mask` | Nodes used for validation |
| `data.test_mask` | Nodes used for testing |

The graph is not stored as a full adjacency matrix. Instead, PyG stores edges efficiently using `edge_index`.

The `edge_index` tensor has shape:

$$
2 \times |E|
$$

The first row stores source nodes, and the second row stores target nodes.

In [4]:
# Code, Dataset Summary

summary = {
    "Number of graphs": len(dataset),
    "Nodes": data.num_nodes,
    "Edges": data.num_edges,
    "Node features": dataset.num_node_features,
    "Classes": dataset.num_classes,
    "Training nodes": int(data.train_mask.sum()),
    "Validation nodes": int(data.val_mask.sum()),
    "Test nodes": int(data.test_mask.sum()),
}

summary

{'Number of graphs': 1,
 'Nodes': 2708,
 'Edges': 10556,
 'Node features': 1433,
 'Classes': 7,
 'Training nodes': 140,
 'Validation nodes': 500,
 'Test nodes': 1000}

In [5]:
# Code, Display Summary as Table
import pandas as pd

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Object", "Value"]
)

summary_df

,Object,Value
0,Number of graphs,1
1,Nodes,2708
2,Edges,10556
3,Node features,1433
4,Classes,7
5,Training nodes,140
6,Validation nodes,500
7,Test nodes,1000


# 7. First Successful PyG Milestone

I successfully loaded the Cora citation graph using PyTorch Geometric on ASU Sol.

The graph contains:

| Object | Value |
|---|---:|
| Nodes | 2,708 |
| Edges | 10,556 |
| Node features | 1,433 |
| Classes | 7 |
| Training nodes | 140 |
| Validation nodes | 500 |
| Test nodes | 1,000 |

Mathematically, the graph is:

$$
G = (V, E)
$$

with:

$$
|V| = 2708
$$

and:

$$
|E| = 10556
$$

The node feature matrix is:

$$
X \in \mathbb{R}^{2708 \times 1433}
$$

The label vector is:

$$
y \in \mathbb{R}^{2708}
$$

This confirms that my Sol PyG environment can load a real graph object successfully.

In [6]:
# Code, Inspect Tensor Shapes

print("=== Tensor Shapes ===")
print("data.x shape:", data.x.shape)
print("data.edge_index shape:", data.edge_index.shape)
print("data.y shape:", data.y.shape)

print("\n=== Masks ===")
print("Training nodes:", int(data.train_mask.sum()))
print("Validation nodes:", int(data.val_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

=== Tensor Shapes ===
data.x shape: torch.Size([2708, 1433])
data.edge_index shape: torch.Size([2, 10556])
data.y shape: torch.Size([2708])

=== Masks ===
Training nodes: 140
Validation nodes: 500
Test nodes: 1000


In [8]:
# Inspect First Few Edges

print("=== First 10 Edges ===")
print(data.edge_index[:, :10])

=== First 10 Edges ===
tensor([[ 633, 1862, 2582,    2,  652,  654,    1,  332, 1454, 1666],
        [   0,    0,    0,    1,    1,    1,    2,    2,    2,    2]])


# 8. Interpreting `edge_index`

The first 10 edges from the Cora graph are:

```text
=== First 10 Edges ===
tensor([[ 633, 1862, 2582,    2,  652,  654,    1,  332, 1454, 1666],
        [   0,    0,    0,    1,    1,    1,    2,    2,    2,    2]])
```

PyTorch Geometric stores graph connections in `edge_index`.

The `edge_index` tensor has two rows:

| Row | Meaning |
|---|---|
| First row | Source nodes |
| Second row | Target nodes |

So each column represents one edge.

For example, the first column is:

```text
[633]
[  0]
```

This means there is an edge from node 633 to node 0:

$$
633 \rightarrow 0
$$

The second column is:

```text
[1862]
[   0]
```

This means there is an edge from node 1862 to node 0:

$$
1862 \rightarrow 0
$$

The first 10 edges can be read as:

| Edge number | Source node | Target node |
|---:|---:|---:|
| 1 | 633 | 0 |
| 2 | 1862 | 0 |
| 3 | 2582 | 0 |
| 4 | 2 | 1 |
| 5 | 652 | 1 |
| 6 | 654 | 1 |
| 7 | 1 | 2 |
| 8 | 332 | 2 |
| 9 | 1454 | 2 |
| 10 | 1666 | 2 |

In the Cora citation graph, these edges represent citation-based connections between papers.

This edge structure is what allows the GCN to pass information between neighboring nodes. In simple terms, a paper does not learn only from its own features. It also receives information from connected papers.


---

#  GCN Model Introduction

# 9. Tiny Graph Convolutional Network

The next step is to create a small two-layer Graph Convolutional Network.

The model structure is:

$$
X \rightarrow \mathrm{GCNConv}_1 \rightarrow \mathrm{ReLU} \rightarrow \mathrm{GCNConv}_2 \rightarrow Z
$$

where:

- $X$ is the node feature matrix.
- $\mathrm{GCNConv}_1$ is the first graph convolution layer.
- $\mathrm{ReLU}$ is the activation function.
- $\mathrm{GCNConv}_2$ is the second graph convolution layer.
- $Z$ is the output score matrix.

The goal is not to train yet. The goal is only to run one forward pass.

In [10]:
# Define TinyGCN

import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class TinyGCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        out = self.conv2(h, edge_index)
        return out

In [11]:
# Create Model

model = TinyGCN(
    input_dim=dataset.num_node_features,
    hidden_dim=16,
    output_dim=dataset.num_classes,
)

model

TinyGCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 7)
)

# 10. Model Architecture Explanation

The model has two GCN layers:

```text
GCNConv(1433, 16)
GCNConv(16, 7)

The first layer changes each node from 1,433 original features into 16 learned features:

1433→16

The second layer changes the 16 learned features into 7 class scores:

16→7

This matches the Cora dataset because Cora has 7 possible classes.

At this point, the model has not learned anything yet. Its weights are still initialized randomly.

In [12]:

# Run Forward Pass

model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)

print("Output shape:", out.shape)
print("First node raw scores:", out[0])

Output shape: torch.Size([2708, 7])
First node raw scores: tensor([ 0.0994,  0.0403,  0.0272,  0.0314,  0.0603,  0.1075, -0.0672])


# 11. Forward Pass Interpretation

The output shape is:

$$
2708 \times 7
$$

This means the GCN produced one row of scores for each of the 2,708 nodes.

Each row has 7 values because there are 7 possible classes.

Mathematically, the output is:

$$
Z = F(G, X)
$$

where:

$$
Z \in \mathbb{R}^{2708 \times 7}
$$

This means the model successfully processed the graph and produced class scores for every node.

Because the model has not been trained yet, these scores are not meaningful predictions. They only confirm that the computation works.

In [13]:
# First Node Prediction Demonstration

first_node_scores = out[0]
predicted_class = int(first_node_scores.argmax())

print("First node scores:")
print(first_node_scores)

print("\nHighest score index:", predicted_class)
print("This is the class the untrained model currently leans toward for node 0.")

First node scores:
tensor([ 0.0994,  0.0403,  0.0272,  0.0314,  0.0603,  0.1075, -0.0672])

Highest score index: 5
This is the class the untrained model currently leans toward for node 0.


# 12. Simple Explanation

A graph is like a network.

For the Cora dataset:

- Each paper is like a student in a school.
- Each citation link is like a connection between students.
- Each paper has 1,433 pieces of information.
- The model tries to produce 7 category scores for each paper.

The GCN does not look at each paper alone. It also looks at connected papers.

So the GCN asks:

> What does this paper look like, and what do its neighboring papers look like?

The output shape was:

$$
2708 \times 7
$$

This means:

> 2,708 papers, each with 7 class scores.

## Example: First Node Output

For node 0, the model produced these raw scores:

```text
tensor([ 0.0994,  0.0403,  0.0272,  0.0314,  0.0603,  0.1075, -0.0672])
```

These are the model's scores for the 7 possible classes.

| Class Index | Raw Score |
|---:|---:|
| 0 | 0.0994 |
| 1 | 0.0403 |
| 2 | 0.0272 |
| 3 | 0.0314 |
| 4 | 0.0603 |
| 5 | 0.1075 |
| 6 | -0.0672 |

The highest score is:

$$
0.1075
$$

This score is at class index:

$$
5
$$

So, for node 0, the untrained model currently leans toward:

```text
Class 5
```

However, this does **not** mean the model is correct yet. The model has not been trained. At this stage, the model is only making a random initial guess based on its randomly initialized weights.

This is why the forward pass is an important first milestone. It confirms that the graph data can move through the GCN model and produce output in the correct shape.

# 13. Connection to Future GNN Stability Research

This first notebook is the starting point for structural stability analysis.

Later, I can compare the original graph:

$$
G
$$

with a perturbed graph:

$$
G'
$$

A perturbed graph may have:

- Removed edges
- Added edges
- Changed node features
- Noisy structure

A future stability question can be written as:

$$
\Delta = \lVert F(G, X) - F(G', X') \rVert
$$

where:

- $F$ is the GNN model.
- $G$ is the original graph.
- $G'$ is the perturbed graph.
- $X$ is the original node feature matrix.
- $X'$ is the perturbed node feature matrix.

If $\Delta$ is small, the model output is relatively stable.

If $\Delta$ is large, the model is sensitive to graph perturbation.

# 14. What I Completed in This First Phase

In this first PyG sandbox phase, I completed the following:

1. Logged into ASU Sol.
2. Worked inside the scratch sandbox directory:

```text
/scratch/shirpa/gnn-stability/research/brainstorming

# 15. Current Limitations

This notebook is only the first technical milestone.

The current limitations are:

1. The model has not been trained yet.
2. No loss function has been optimized yet.
3. No accuracy has been measured yet.
4. No perturbation experiment has been performed yet.
5. No stability metric has been calculated yet.
6. No comparison across random seeds has been done yet.

These are not problems. They define the next steps.

The purpose of this notebook is only to confirm that the PyG graph workflow works on Sol.

# 16. Next Steps

The next phase will build on this notebook.

## Phase 2: Train the GCN

The next notebook should add:

- Training loop
- Cross-entropy loss
- Optimizer
- Train accuracy
- Validation accuracy
- Test accuracy

## Phase 3: Beginner Stability Test

After training works, I can create a simple perturbation experiment:

- Remove 5% of edges
- Remove 10% of edges
- Remove 20% of edges
- Compare clean accuracy with perturbed accuracy

A simple beginner stability metric will be:

$$
\Delta_{\text{accuracy}}
=
\text{Accuracy}_{clean}
-
\text{Accuracy}_{perturbed}
$$

## Phase 4: Research Upgrade

Later, this can be upgraded toward Professor He's research direction by studying:

- Structural perturbations
- Stochastic graph changes
- Output sensitivity
- Lipschitz-type behavior
- Fixed-point inspired stability ideas
- Reproducible experiments on Sol

# 17. Conclusion

This notebook documents my first successful PyTorch Geometric milestone on ASU Sol.

I successfully loaded the Cora citation graph, inspected its graph structure, created a small two-layer GCN, and ran one forward pass.

The most important result is:

$$
F(G, X) \in \mathbb{R}^{2708 \times 7}
$$

This means the GCN processed the Cora graph and produced 7 class scores for each of the 2,708 nodes.

This first phase confirms that my Sol sandbox, Conda environment, Jupyter kernel, PyTorch, and PyTorch Geometric setup are working correctly.

This is a small but important foundation for future work on GNN training, perturbation experiments, and structural stability analysis.